# Diffusion and flow matching

Train two generative processors with Lightning, then change their sampling settings without retraining. This notebook runs on its own; it reuses the small AutoSim dataset and cached latents from [Autoencoder and latent data](autoencoder_and_latents.ipynb), or prepares them below.

| Processor | Learns | Inference control |
| --- | --- | --- |
| Flow matching | A velocity field between noise and data | `flow_ode_steps` |
| Diffusion | A denoising map across a noise schedule | `sampler_steps` |

The backbone is interchangeable; the processor defines the objective and sampler.

In [ ]:
import json

import lightning as L
import matplotlib.pyplot as plt
import torch
from _support import OUTPUT_ROOT, prepare_tutorial
from azula.noise import VPSchedule
from hydra.utils import instantiate
from omegaconf import OmegaConf

from autocast.data.encoded_dataset import CachedLatentDataset, EncodedDataModule
from autocast.models.autoencoder import AE
from autocast.models.processor import ProcessorModel
from autocast.nn.unet import TemporalUNetBackbone
from autocast.nn.vit import TemporalViTBackbone
from autocast.processors.diffusion import DiffusionProcessor
from autocast.processors.flow_matching import FlowMatchingProcessor
from autocast.utils import get_optimizer_config
from autocast.utils.plots import plot_spatiotemporal_snapshots

torch.set_num_threads(2)
L.seed_everything(42, workers=True)

## Prepare the example

Reuse an existing autoencoder and cache. On a fresh start, this runs the tiny autoencoder tutorial in a separate kernel (about 20 seconds on a local CPU); the training code lives there, not in a duplicate helper.

In [ ]:
prepare_tutorial("autoencoder_and_latents")

## Load cached windows

Both processors see two input frames and predict the next two. Each latent frame is an 8×8 grid with two channels; `global_cond` holds the simulator parameters.

In [ ]:
latent_dir = OUTPUT_ROOT / "autoencoder" / "cached_latents"
window_options = {"n_steps_input": 2, "n_steps_output": 2, "stride": 2}
data_options = dict(
    data_path=str(latent_dir), batch_size=4, num_workers=0, **window_options
)
datamodule = EncodedDataModule(dataset_cls=CachedLatentDataset, **data_options)
batch = next(iter(datamodule.train_dataloader()))
channels = batch.encoded_inputs.shape[-1]

batch.encoded_inputs.shape, batch.encoded_output_fields.shape

## Choose a backbone

Set `backbone` to `"unet"` or `"vit"`, then rerun from here. Both are deliberately small CPU examples. Each processor gets its own backbone instance and weights.

In [ ]:
backbone = "unet"  # or "vit"
backbone_class, architecture = {
    "unet": (TemporalUNetBackbone, {"hid_channels": (8, 16), "hid_blocks": (1, 1)}),
    "vit": (
        TemporalViTBackbone,
        {"hid_channels": 32, "hid_blocks": 2, "attention_heads": 4, "patch_size": 1},
    ),
}[backbone]
backbone_options = dict(
    in_channels=channels,
    out_channels=channels,
    cond_channels=channels,
    n_steps_input=window_options["n_steps_input"],
    n_steps_output=window_options["n_steps_output"],
    global_cond_channels=batch.global_cond.shape[-1],
    include_global_cond=True,
    mod_features=32,
    **architecture,
)
processor_options = {
    "n_steps_output": window_options["n_steps_output"],
    "n_channels_out": channels,
}

In [ ]:
processors = {
    "Flow matching": FlowMatchingProcessor(
        backbone=backbone_class(**backbone_options), **processor_options
    ),
    "Diffusion": DiffusionProcessor(
        backbone=backbone_class(**backbone_options),
        schedule=VPSchedule(),
        **processor_options,
    ),
}

## Train with Lightning

`ProcessorModel` delegates the loss to the chosen processor. Ten epochs are enough to exercise the workflow, not to rank models. The two training losses have different meanings; compare decoded forecasts instead.

In [ ]:
suffix = {"unet": "", "vit": "_vit"}[backbone]
run_dirs = {
    "Flow matching": OUTPUT_ROOT / "generative_processors" / f"flow_matching{suffix}",
    "Diffusion": OUTPUT_ROOT / "generative_processors" / f"diffusion{suffix}",
}
optimizer = get_optimizer_config(learning_rate=3e-3)
models = {}
for name, processor in processors.items():
    model = ProcessorModel(
        processor=processor, stride=window_options["stride"], optimizer_config=optimizer
    )
    trainer = L.Trainer(
        max_epochs=10,
        accelerator="cpu",
        devices=1,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    trainer.fit(model, datamodule=datamodule)
    models[name] = model.eval()
    run_dirs[name].mkdir(parents=True, exist_ok=True)
    trainer.save_checkpoint(run_dirs[name] / "processor.ckpt")

## Restore the autoencoder

Use the architecture and raw-data settings saved alongside the cache. The reference trajectory below contains original physical fields, not decoded latents, so the comparison includes autoencoder error.

In [ ]:
ae_config = OmegaConf.load(latent_dir / "autoencoder_config.yaml")
autoencoder = AE.load_from_checkpoint(
    ae_config.autoencoder_checkpoint,
    encoder=instantiate(ae_config.model.encoder),
    decoder=instantiate(ae_config.model.decoder),
    map_location="cpu",
).eval()
raw_datamodule = instantiate(
    ae_config.datamodule,
    autoencoder_mode=False,
    full_trajectory_mode=False,
    **window_options,
)
raw_batch = next(iter(raw_datamodule.rollout_test_dataloader(batch_size=1)))
with torch.no_grad():
    encoded_batch = autoencoder.encoder.encode_batch(raw_batch)

## Sample and plot

Change `seed` or `inference_steps` and rerun these two cells—no training required. Roll forward twice in latent space, then decode four forecast frames. More sampling steps cost more; they do not guarantee a better forecast from these short training runs.

In [ ]:
seed = 7
inference_steps = 4
processors["Flow matching"].flow_ode_steps = inference_steps
processors["Diffusion"].sampler_steps = inference_steps

forecasts = {}
with torch.no_grad():
    for name, model in models.items():
        torch.manual_seed(seed)
        latent_prediction, _ = model.rollout(
            encoded_batch,
            stride=window_options["stride"],
            max_rollout_steps=2,
            free_running_only=True,
        )
        forecasts[name] = autoencoder.decode(latent_prediction)
truth = raw_batch.output_fields[:, : latent_prediction.shape[1]]

In [ ]:
for name, prediction in forecasts.items():
    figure = plot_spatiotemporal_snapshots(
        true=truth,
        pred=prediction,
        timesteps=range(truth.shape[1]),
        title=name,
    )
    plt.show()

## Record runs for evaluation

Save the construction settings beside each checkpoint so the CLI and [evaluation notebook](evaluation_and_results.ipynb) can reload these Python-trained models. Rerun this cell after changing sampling settings.

In [ ]:
backbone_config = {
    "_target_": f"{backbone_class.__module__}.{backbone_class.__name__}",
    **backbone_options,
}
sampling_configs = {
    "Flow matching": {"flow_ode_steps": processors["Flow matching"].flow_ode_steps},
    "Diffusion": {
        "sampler_steps": processors["Diffusion"].sampler_steps,
        "schedule": {"_target_": "azula.noise.VPSchedule"},
    },
}
for name, processor in processors.items():
    processor_class = type(processor)
    config = OmegaConf.create(
        {
            "datamodule": {
                "_target_": "autocast.data.encoded_dataset.EncodedDataModule",
                "dataset_cls": {
                    "_target_": "hydra.utils.get_class",
                    "path": "autocast.data.encoded_dataset.CachedLatentDataset",
                },
                **data_options,
            },
            "model": {
                "processor": {
                    "_target_": (
                        f"{processor_class.__module__}.{processor_class.__name__}"
                    ),
                    "backbone": backbone_config,
                    **processor_options,
                    **sampling_configs[name],
                }
            },
            "optimizer": optimizer,
            "output": {"save_config": True, "checkpoint_name": "processor.ckpt"},
        }
    )
    OmegaConf.save(config, run_dirs[name] / "resolved_config.yaml")

selected_runs = {
    name: str(path.relative_to(OUTPUT_ROOT)) for name, path in run_dirs.items()
}
(OUTPUT_ROOT / "processor_runs.json").write_text(json.dumps(selected_runs))
selected_runs

Next, try [deterministic ensembles](deterministic_ensembles.ipynb) for the separate noise-conditioning approach. [Evaluation and results](evaluation_and_results.ipynb) compares physical-space metrics and uncertainty across the saved runs.